In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/yelp-dataset/yelp-dataset/Dataset_User_Agreement.pdf
/kaggle/input/datasets/yelp-dataset/yelp-dataset/yelp_academic_dataset_business.json
/kaggle/input/datasets/yelp-dataset/yelp-dataset/yelp_academic_dataset_checkin.json
/kaggle/input/datasets/yelp-dataset/yelp-dataset/yelp_academic_dataset_review.json
/kaggle/input/datasets/yelp-dataset/yelp-dataset/yelp_academic_dataset_tip.json
/kaggle/input/datasets/yelp-dataset/yelp-dataset/yelp_academic_dataset_user.json


## 1. Thiết lập môi trường & Kéo mã nguồn dự án

Cell code ngay bên dưới thực hiện clone (hoặc pull cập nhật) repository `toan58936/topic` về `/kaggle/working/topic`, thêm đường dẫn vào `sys.path` để import các module trong `src/` (`data_setup`, `baseline`, `metrics`, ...), và chuyển thư mục làm việc hiện tại sang thư mục repo.

In [2]:
from src.data_setup import prepare_splits

# Tu dong tim file review.json trong /kaggle/input (tranh hardcode sai duong dan)
RAW_JSON = None
for dirname, _, filenames in os.walk("/kaggle/input"):
    for fname in filenames:
        if fname == "review.json":
            RAW_JSON = os.path.join(dirname, fname)
            break
    if RAW_JSON:
        break
if RAW_JSON is None:
    raise FileNotFoundError("Khong tim thay review.json trong /kaggle/input. Kiem tra Dataset da gan chua.")
DATA_DIR = "/kaggle/working/data"

stats = prepare_splits(
    review_path=RAW_JSON,
    output_dir=DATA_DIR,
    n_samples=60000,
    ratios=(0.70, 0.15, 0.15),
    seed=42,
    overwrite=True,
)

print("Split ID:", stats["split_id"])
print("So luong mau (Train / Val / Test):", stats["train_size"], stats["val_size"], stats["test_size"])
print("So mau trung text bi loai:", stats["duplicate_text_removed"])
print("Phan phoi sao tap Test:", stats["stars_per_split"]["test"])

# --- Validation schema split_stats.json (plans/phase1.md §1.3) ---
import json as _json
from pathlib import Path as _Path
stats_path = _Path(DATA_DIR) / "split_stats.json"
with open(stats_path, "r", encoding="utf-8") as f:
    saved_stats = _json.load(f)

required_keys = ["split_id", "seed", "n_samples", "train_size", "val_size", "test_size",
                 "stars_per_split", "stars_distribution_original", "duplicate_text_removed",
                 "user_id_overlap_rate", "business_id_overlap_rate", "created_at"]
missing = [k for k in required_keys if k not in saved_stats]
if missing:
    raise ValueError(f"split_stats.json thieu cac khoa: {missing}")

assert len(saved_stats["split_id"]) == 12, f"split_id phai 12 ky tu, nhan {len(saved_stats['split_id'])}"
assert saved_stats["seed"] == 42, f"seed phai 42, nhan {saved_stats['seed']}"
assert saved_stats["n_samples"] == 60000, f"n_samples phai 60000, nhan {saved_stats['n_samples']}"
assert saved_stats["train_size"] == 42000, f"train_size phai 42000, nhan {saved_stats['train_size']}"
assert saved_stats["val_size"] == 9000, f"val_size phai 9000, nhan {saved_stats['val_size']}"
assert saved_stats["test_size"] == 9000, f"test_size phai 9000, nhan {saved_stats['test_size']}"
for split in ("train", "val", "test"):
    for star in ("1", "2", "3", "4", "5"):
        assert str(star) in saved_stats["stars_per_split"][split], f"Thieu stars_per_split[{split}][{star}]"

# Luu split_id vao file metadata de Phase 5 de doc
split_id_path = _Path(DATA_DIR) / "split_id.json"
with open(split_id_path, "w", encoding="utf-8") as f:
    _json.dump({"split_id": saved_stats["split_id"]}, f, ensure_ascii=False, indent=2)

print("✅ split_stats.json schema hop le")
print("💾 Da luu split_id vao", split_id_path)


Cloning into '/kaggle/working/topic'...
remote: Enumerating objects: 334, done.
remote: Counting objects: 100% (334/334), done.
remote: Compressing objects: 100% (235/235), done.
remote: Total 334 (delta 166), reused 248 (delta 96), pack-reused 0 (from 0)
Receiving objects: 100% (334/334), 345.58 KiB | 1.33 MiB/s, done.
Resolving deltas: 100% (166/166), done.
Thư mục hiện tại: /kaggle/working/topic


## 2. Chia dữ liệu Train / Val / Test (`prepare_splits`)

Cell code ngay bên dưới đọc dữ liệu review Yelp gốc, chuẩn hóa và loại bỏ trùng lặp theo quy tắc cấu hình, sau đó chia ngẫu nhiên có tái lập (`seed=42`) thành `42,000` Train / `9,001` Val / `8,999` Test (`Split ID: f0e6776f8a1a`) và lưu các file `.parquet` cùng `split_stats.json` vào `/kaggle/working/data`.

In [3]:
from src.data_setup import prepare_splits

RAW_JSON = "/kaggle/input/datasets/yelp-dataset/yelp-dataset/yelp_academic_dataset_review.json"
DATA_DIR = "/kaggle/working/data"

stats = prepare_splits(
    review_path=RAW_JSON,
    output_dir=DATA_DIR,
    n_samples=60000,
    ratios=(0.70, 0.15, 0.15),
    seed=42,
    overwrite=True,
)

print("Split ID:", stats["split_id"])
print("Số lượng mẫu (Train / Val / Test):", stats["train_size"], stats["val_size"], stats["test_size"])
print("Số mẫu trùng text bị loại:", stats["duplicate_text_removed"])
print("Phân phối sao tập Test:", stats["stars_per_split"]["test"])

Split ID: f0e6776f8a1a
Số lượng mẫu (Train / Val / Test): 42000 9001 8999
Số mẫu trùng text bị loại: {'val': 0, 'test': 0}
Phân phối sao tập Test: {'1': 1377, '2': 700, '3': 891, '4': 1870, '5': 4161}


## 3. Khám phá dữ liệu (EDA) trên các tập Train / Val / Test

Các cell code tiếp theo phân tích dữ liệu đã chia tại `/kaggle/working/data` (`train.parquet`, `val.parquet`, `test.parquet` và `split_stats.json`) nhằm đánh giá:
* Phân phối nhãn `stars` (`1`–`5`) và mức độ đồng đều giữa các tập
* Độ dài văn bản (ký tự, từ) và tỷ lệ review vượt ngưỡng `256` token của Transformer
* Mối liên hệ giữa độ dài review và điểm sao `stars`
* Độ sạch, trùng lặp nội dung/định danh và mức độ lặp của `user_id` / `business_id`
* Các từ và cụm 2 từ (`1-gram`, `2-gram`) đặc trưng theo từng mức sao

In [4]:
import json
import numpy as np
import pandas as pd

DATA_DIR = "/kaggle/working/data"

train_df = pd.read_parquet(f"{DATA_DIR}/train.parquet")
val_df = pd.read_parquet(f"{DATA_DIR}/val.parquet")
test_df = pd.read_parquet(f"{DATA_DIR}/test.parquet")

with open(f"{DATA_DIR}/split_stats.json", "r", encoding="utf-8") as f:
    split_stats = json.load(f)

splits = {"train": train_df, "val": val_df, "test": test_df}
star_levels = [1, 2, 3, 4, 5]

# 1. Số lượng và tỷ lệ % từng mức sao (1..5)
counts = pd.DataFrame(
    {
        name: df["stars"].astype(int).value_counts().reindex(star_levels, fill_value=0)
        for name, df in splits.items()
    }
)
counts.loc["Tổng"] = counts.sum()

pcts = counts.iloc[:5].div(counts.iloc[:5].sum(axis=0), axis=1).mul(100).round(2)
pcts.loc["Tổng %"] = pcts.sum().round(2)

print(f"Split ID: {split_stats.get('split_id')}")
print("\n--- Số lượng mẫu theo mức sao ---")
display(counts)

print("\n--- Tỷ lệ phần trăm (%) theo mức sao ---")
display(pcts)

# 2. Trung bình, độ lệch chuẩn, trung vị, IQR, độ lệch phân phối
summary = pd.DataFrame(
    [
        {
            "split": name,
            "n": len(df),
            "mean": round(float(df["stars"].mean()), 4),
            "std": round(float(df["stars"].std(ddof=1)), 4),
            "median": float(df["stars"].median()),
            "iqr": float(df["stars"].quantile(0.75) - df["stars"].quantile(0.25)),
            "skew": round(float(df["stars"].skew()), 4),
        }
        for name, df in splits.items()
    ]
).set_index("split")

print("\n--- Thống kê tổng quan stars ---")
display(summary)

# 3. Độ đồng đều giữa các tập (chênh lệch max - min về tỷ lệ %)
max_diff = {
    f"star_{k}": round(float(pcts.loc[k].max() - pcts.loc[k].min()), 2)
    for k in star_levels
}
print("\n--- Chênh lệch tỷ lệ (%) lớn nhất giữa các tập ---")
print(max_diff)

# 4. Minh họa lý do mô hình thiên về vùng điểm phổ biến dưới sai số bình phương
train_mean = float(train_df["stars"].mean())
val_y = val_df["stars"].to_numpy(dtype=float)
rmse_mean = float(np.sqrt(np.mean((val_y - train_mean) ** 2)))
print(
    f"\nDự đoán hằng số = mean(train) ({train_mean:.4f}) trên Val -> "
    f"RMSE = {rmse_mean:.4f}"
)

Split ID: f0e6776f8a1a

--- Số lượng mẫu theo mức sao ---


,train,val,test
stars,,,
1,6427,1377,1377
2,3270,701,700
3,4157,891,891
4,8730,1871,1870
5,19416,4161,4161
Tổng,42000,9001,8999



--- Tỷ lệ phần trăm (%) theo mức sao ---


,train,val,test
stars,,,
1,15.30,15.30,15.30
2,7.79,7.79,7.78
3,9.90,9.90,9.90
4,20.79,20.79,20.78
5,46.23,46.23,46.24
Tổng %,100.01,100.01,100.00



--- Thống kê tổng quan stars ---


,n,mean,std,median,iqr,skew
split,,,,,,
train,42000,3.7485,1.4788,4.0,2.0,-0.8430
val,9001,3.7486,1.4787,4.0,2.0,-0.8431
test,8999,3.7487,1.4788,4.0,2.0,-0.8434



--- Chênh lệch tỷ lệ (%) lớn nhất giữa các tập ---
{'star_1': 0.0, 'star_2': 0.01, 'star_3': 0.0, 'star_4': 0.01, 'star_5': 0.01}

Dự đoán hằng số = mean(train) (3.7485) trên Val -> RMSE = 1.4787


In [5]:
import numpy as np
import pandas as pd

DATA_DIR = "/kaggle/working/data"

splits = {
    "train": pd.read_parquet(f"{DATA_DIR}/train.parquet"),
    "val": pd.read_parquet(f"{DATA_DIR}/val.parquet"),
    "test": pd.read_parquet(f"{DATA_DIR}/test.parquet"),
}

quantiles = [0.50, 0.90, 0.95, 0.99]
char_rows = []
word_rows = []
trunc_rows = []

for name, df in splits.items():
    texts = df["text"].astype(str)
    char_lens = texts.str.len()
    word_lens = pd.Series([len(t.split()) for t in texts], index=texts.index)

    # Ước lượng token: 1 từ tiếng Anh trung bình ~ 1.3 token (hoặc xấp xỉ theo số từ)
    est_tokens_13 = word_lens * 1.3

    char_rows.append({
        "split": name,
        "mean": round(float(char_lens.mean()), 2),
        "median_p50": round(float(char_lens.quantile(0.50)), 2),
        "p90": round(float(char_lens.quantile(0.90)), 2),
        "p95": round(float(char_lens.quantile(0.95)), 2),
        "p99": round(float(char_lens.quantile(0.99)), 2),
        "max": int(char_lens.max()),
    })

    word_rows.append({
        "split": name,
        "mean": round(float(word_lens.mean()), 2),
        "median_p50": round(float(word_lens.quantile(0.50)), 2),
        "p90": round(float(word_lens.quantile(0.90)), 2),
        "p95": round(float(word_lens.quantile(0.95)), 2),
        "p99": round(float(word_lens.quantile(0.99)), 2),
        "max": int(word_lens.max()),
    })

    trunc_rows.append({
        "split": name,
        "words_gt_256_pct": round(float((word_lens > 256).mean() * 100), 2),
        "est_tokens_1.3x_gt_256_pct": round(float((est_tokens_13 > 256).mean() * 100), 2),
        "est_tokens_1.3x_gt_254_pct": round(float((est_tokens_13 > 254).mean() * 100), 2),
    })

print("--- ĐỘ DÀI THEO SỐ KÝ TỰ ---")
display(pd.DataFrame(char_rows).set_index("split"))

print("\n--- ĐỘ DÀI THEO SỐ TỪ (WHITESPACE SPLIT) ---")
display(pd.DataFrame(word_rows).set_index("split"))

print("\n--- ƯỚC LƯỢNG TỶ LỆ REVIEW VƯỢT NGƯỠNG 256 TOKEN (TÍNH CẢ TOKEN ĐẶC BIỆT) ---")
display(pd.DataFrame(trunc_rows).set_index("split"))

--- ĐỘ DÀI THEO SỐ KÝ TỰ ---


,mean,median_p50,p90,p95,p99,max
split,,,,,,
train,566.32,402.0,1175.0,1569.0,2598.04,4990
val,564.88,406.0,1175.0,1536.0,2581.00,4987
test,554.46,395.0,1160.2,1541.2,2479.06,4965



--- ĐỘ DÀI THEO SỐ TỪ (WHITESPACE SPLIT) ---


,mean,median_p50,p90,p95,p99,max
split,,,,,,
train,105.06,74.0,219.0,293.0,488.0,1003
val,104.89,75.0,217.0,292.0,480.0,967
test,102.74,74.0,214.0,284.0,471.0,952



--- ƯỚC LƯỢNG TỶ LỆ REVIEW VƯỢT NGƯỠNG 256 TOKEN (TÍNH CẢ TOKEN ĐẶC BIỆT) ---


,words_gt_256_pct,est_tokens_1.3x_gt_256_pct,est_tokens_1.3x_gt_254_pct
split,,,
train,6.98,12.56,12.71
val,6.93,12.57,12.67
test,6.58,12.13,12.30


In [6]:
import numpy as np
import pandas as pd

DATA_DIR = "/kaggle/working/data"

train_df = pd.read_parquet(f"{DATA_DIR}/train.parquet")
val_df = pd.read_parquet(f"{DATA_DIR}/val.parquet")
test_df = pd.read_parquet(f"{DATA_DIR}/test.parquet")

# Gộp toàn bộ 60,000 mẫu của split hiện tại (hoặc đổi sang train_df nếu muốn chỉ tính trên Train)
full_df = pd.concat(
    [
        train_df.assign(split="train"),
        val_df.assign(split="val"),
        test_df.assign(split="test"),
    ],
    ignore_index=True,
)
full_df["stars_int"] = full_df["stars"].astype(int)
full_df["word_len"] = full_df["text"].astype(str).str.split().str.len()
full_df["char_len"] = full_df["text"].astype(str).str.len()

# 1. Thống kê độ dài theo từng mức sao (1..5) trên toàn bộ 60,000 mẫu
rows = []
for star, grp in full_df.groupby("stars_int"):
    w = grp["word_len"]
    c = grp["char_len"]
    rows.append({
        "stars": star,
        "n": len(grp),
        "word_mean": round(float(w.mean()), 2),
        "word_median": round(float(w.median()), 1),
        "word_std": round(float(w.std(ddof=1)), 2),
        "word_iqr": round(float(w.quantile(0.75) - w.quantile(0.25)), 1),
        "word_p90": round(float(w.quantile(0.90)), 1),
        "word_max": int(w.max()),
        "char_mean": round(float(c.mean()), 1),
        "char_median": round(float(c.median()), 1),
        "char_std": round(float(c.std(ddof=1)), 1),
        "char_iqr": round(float(c.quantile(0.75) - c.quantile(0.25)), 1),
    })

by_star_df = pd.DataFrame(rows).set_index("stars")
print("--- ĐỘ DÀI REVIEW (SỐ TỪ & SỐ KÝ TỰ) THEO TỪNG MỨC SAO ---")
display(by_star_df)

# 2. So sánh nhóm chê (1-2 sao), trung lập (3 sao), khen (4-5 sao)
def group_label(s: int) -> str:
    if s <= 2:
        return "Chê (1-2 sao)"
    if s == 3:
        return "Trung lập (3 sao)"
    return "Khen (4-5 sao)"

full_df["sentiment_group"] = full_df["stars_int"].map(group_label)
grp_summary = (
    full_df.groupby("sentiment_group")
    .agg(
        n=("stars_int", "count"),
        word_mean=("word_len", "mean"),
        word_median=("word_len", "median"),
        word_std=("word_len", "std"),
        word_iqr=("word_len", lambda x: x.quantile(0.75) - x.quantile(0.25)),
        char_mean=("char_len", "mean"),
        char_median=("char_len", "median"),
    )
    .round(2)
)
print("\n--- SO SÁNH NHÓM CHÊ / TRUNG LẬP / KHEN ---")
display(grp_summary)

# 3. Hệ số tương quan giữa độ dài và điểm sao (Pearson & Spearman)
pearson_w = float(full_df["word_len"].corr(full_df["stars"], method="pearson"))
spearman_w = float(full_df["word_len"].corr(full_df["stars"], method="spearman"))
pearson_c = float(full_df["char_len"].corr(full_df["stars"], method="pearson"))
spearman_c = float(full_df["char_len"].corr(full_df["stars"], method="spearman"))

print("\n--- TƯƠNG QUAN ĐỘ DÀI VÀ ĐIỂM SAO ---")
print(f"Số từ vs stars : Pearson r = {pearson_w:.4f} | Spearman rho = {spearman_w:.4f}")
print(f"Số ký tự vs stars: Pearson r = {pearson_c:.4f} | Spearman rho = {spearman_c:.4f}")

--- ĐỘ DÀI REVIEW (SỐ TỪ & SỐ KÝ TỰ) THEO TỪNG MỨC SAO ---


,n,word_mean,word_median,word_std,word_iqr,word_p90,word_max,char_mean,char_median,char_std,char_iqr
stars,,,,,,,,,,,
1,9181,134.59,96.0,125.48,116.0,281.0,1003,713.4,512.0,661.4,612.0
2,4671,133.28,100.0,115.02,114.0,274.0,973,712.1,534.0,612.8,607.0
3,5939,123.97,95.0,102.93,107.0,250.0,937,664.3,508.0,551.7,568.5
4,12471,108.38,81.0,94.26,95.0,225.0,967,586.2,435.0,509.0,512.0
5,27738,84.18,60.0,78.00,70.0,173.0,951,458.9,329.0,422.1,376.8



--- SO SÁNH NHÓM CHÊ / TRUNG LẬP / KHEN ---


,n,word_mean,word_median,word_std,word_iqr,char_mean,char_median
sentiment_group,,,,,,,
Chê (1-2 sao),13852,134.15,97.0,122.05,115.0,712.98,518.0
Khen (4-5 sao),40209,91.68,66.0,84.13,79.0,498.35,357.0
Trung lập (3 sao),5939,123.97,95.0,102.93,107.0,664.30,508.0



--- TƯƠNG QUAN ĐỘ DÀI VÀ ĐIỂM SAO ---
Số từ vs stars : Pearson r = -0.2043 | Spearman rho = -0.2300
Số ký tự vs stars: Pearson r = -0.1940 | Spearman rho = -0.2199


In [7]:
import json
import pandas as pd

DATA_DIR = "/kaggle/working/data"

splits = {
    "train": pd.read_parquet(f"{DATA_DIR}/train.parquet"),
    "val": pd.read_parquet(f"{DATA_DIR}/val.parquet"),
    "test": pd.read_parquet(f"{DATA_DIR}/test.parquet"),
}

with open(f"{DATA_DIR}/split_stats.json", "r", encoding="utf-8") as f:
    split_stats = json.load(f)

target_cols = ["text", "review_id", "user_id", "business_id", "stars"]

# 1. Kiểm tra null, chuỗi rỗng "", chuỗi chỉ có khoảng trắng (whitespace-only)
clean_rows = []
for name, df in splits.items():
    row = {"split": name, "n_rows": len(df)}
    for col in target_cols:
        if col not in df.columns:
            row[f"{col}_missing_col"] = True
            continue
        s = df[col]
        n_null = int(s.isna().sum())
        if s.dtype == object or pd.api.types.is_string_dtype(s):
            as_str = s.dropna().astype(str)
            n_empty = int((as_str.str.len() == 0).sum())
            n_ws_only = int(((as_str.str.len() > 0) & (as_str.str.strip().str.len() == 0)).sum())
        else:
            n_empty = 0
            n_ws_only = 0
        row[f"{col}_null"] = n_null
        row[f"{col}_empty"] = n_empty
        row[f"{col}_ws_only"] = n_ws_only
    clean_rows.append(row)

print("--- 1. KIỂM TRA NULL / RỖNG / CHỈ CÓ KHOẢNG TRẮNG ---")
display(pd.DataFrame(clean_rows).set_index("split"))

# 2. Trùng lặp nội dung text (nguyên bản & chuẩn hóa thường/gọn khoảng trắng) và review_id
dup_rows = []
norm_sets = {}
raw_sets = {}
rid_sets = {}

for name, df in splits.items():
    raw_text = df["text"].astype(str)
    norm_text = raw_text.str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
    rids = df["review_id"].astype(str)

    raw_sets[name] = set(raw_text)
    norm_sets[name] = set(norm_text)
    rid_sets[name] = set(rids)

    dup_rows.append({
        "split": name,
        "unique_review_id": int(rids.nunique()),
        "dup_review_id_in_split": int(rids.duplicated().sum()),
        "unique_raw_text": int(raw_text.nunique()),
        "dup_raw_text_in_split": int(raw_text.duplicated().sum()),
        "unique_norm_text": int(norm_text.nunique()),
        "dup_norm_text_in_split": int(norm_text.duplicated().sum()),
    })

print("\n--- 2A. TRÙNG LẶP TRONG TỪNG TẬP ---")
display(pd.DataFrame(dup_rows).set_index("split"))

# Giao thoa (overlap) giữa các cặp tập theo norm_text, raw_text và review_id
pairs = [("train", "val"), ("train", "test"), ("val", "test")]
cross_rows = []
for a, b in pairs:
    cross_rows.append({
        "pair": f"{a} ∩ {b}",
        "overlap_norm_text": len(norm_sets[a] & norm_sets[b]),
        "overlap_raw_text": len(raw_sets[a] & raw_sets[b]),
        "overlap_review_id": len(rid_sets[a] & rid_sets[b]),
    })

print("\n--- 2B. GIAO THOA GIỮA CÁC TẬP (NORM TEXT / RAW TEXT / REVIEW_ID) ---")
display(pd.DataFrame(cross_rows).set_index("pair"))

print("Đối chiếu split_stats.json -> duplicate_text_removed:", split_stats.get("duplicate_text_removed"))

# 3. Mức độ xuất hiện lặp lại của user_id và business_id (trên toàn bộ 60,000 mẫu của split)
full_df = pd.concat(
    [df.assign(split=name) for name, df in splits.items()],
    ignore_index=True,
)

id_summary_rows = []
for id_col in ["user_id", "business_id"]:
    vc = full_df[id_col].astype(str).value_counts()
    n_total = len(full_df)
    n_unique = len(vc)
    n_gt1 = int((vc > 1).sum())
    n_gt5 = int((vc > 5).sum())
    rows_in_gt1 = int(vc[vc > 1].sum())
    top1_count = int(vc.iloc[0])
    top10_share_pct = round(float(vc.head(10).sum() / n_total * 100), 2)

    id_summary_rows.append({
        "id_col": id_col,
        "total_reviews": n_total,
        "unique_ids": n_unique,
        "reviews_per_id_mean": round(float(vc.mean()), 2),
        "reviews_per_id_median": round(float(vc.median()), 1),
        "reviews_per_id_max": top1_count,
        "ids_with_gt1_review": n_gt1,
        "pct_ids_with_gt1": round(float(n_gt1 / n_unique * 100), 2),
        "pct_reviews_from_gt1_ids": round(float(rows_in_gt1 / n_total * 100), 2),
        "ids_with_gt5_reviews": n_gt5,
        "top10_ids_share_pct": top10_share_pct,
    })

print("\n--- 3A. TỔNG QUAN MỨC ĐỘ LẶP CỦA USER_ID & BUSINESS_ID (60,000 MẪU) ---")
display(pd.DataFrame(id_summary_rows).set_index("id_col"))

for id_col in ["user_id", "business_id"]:
    vc = full_df[id_col].astype(str).value_counts().head(10)
    top_df = pd.DataFrame({
        id_col: vc.index,
        "review_count": vc.values,
        "share_pct_60k": (vc.values / len(full_df) * 100).round(3),
    })
    print(f"\n--- TOP 10 {id_col.upper()} NHIỀU REVIEW NHẤT ---")
    display(top_df.set_index(id_col))

--- 1. KIỂM TRA NULL / RỖNG / CHỈ CÓ KHOẢNG TRẮNG ---


,n_rows,text_null,text_empty,text_ws_only,review_id_null,review_id_empty,review_id_ws_only,user_id_null,user_id_empty,user_id_ws_only,business_id_null,business_id_empty,business_id_ws_only,stars_null,stars_empty,stars_ws_only
split,,,,,,,,,,,,,,,,
train,42000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
val,9001,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
test,8999,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0



--- 2A. TRÙNG LẶP TRONG TỪNG TẬP ---


,unique_review_id,dup_review_id_in_split,unique_raw_text,dup_raw_text_in_split,unique_norm_text,dup_norm_text_in_split
split,,,,,,
train,42000,0,41999,1,41999,1
val,9001,0,9001,0,9001,0
test,8999,0,8999,0,8999,0



--- 2B. GIAO THOA GIỮA CÁC TẬP (NORM TEXT / RAW TEXT / REVIEW_ID) ---


,overlap_norm_text,overlap_raw_text,overlap_review_id
pair,,,
train ∩ val,0,0,0
train ∩ test,0,0,0
val ∩ test,0,0,0


... (2 outputs left out)


--- TOP 10 USER_ID NHIỀU REVIEW NHẤT ---


,review_count,share_pct_60k
user_id,,
_BcWyKQL16ndpBdggh2kNA,25,0.042
Xw7ZjaGfr0WNVt6s_5KZfA,22,0.037
fr1Hz2acAb3OaL3l6DyKNg,18,0.030
1HM81n6n4iPIFU5d2Lokhw,16,0.027
lAW03ccfMEuRZiDqvVgc0w,14,0.023
ET8n-r7glWYqZhuR6GcdNw,14,0.023
0Igx-a1wAstiBDerGxXk2A,13,0.022
eTvp_hYnsrI5-ow_sQ31_g,13,0.022
I2XpWCHAom1JRyHXZQrnfg,13,0.022



--- TOP 10 BUSINESS_ID NHIỀU REVIEW NHẤT ---


,review_count,share_pct_60k
business_id,,
ac1AeYqs8Z4_e2X5M3if2A,66,0.110
ytynqOUb3hjKeJfRj5Tshw,59,0.098
_C7QiQQc47AOEv4PE3Kong,55,0.092
_ab50qdWOk0DdB6XOrBitw,55,0.092
GXFMD0Z4jEVZBCsbPf4CTQ,52,0.087
oBNrLz4EDhiscSlbOl8uAw,48,0.080
VaO-VW3e1kARkU9bP1E7Fw,42,0.070
iSRTaT9WngzB8JJ2YKJUig,39,0.065
6a4gLLFSgr-Q6CZXDLzBGQ,38,0.063


In [8]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

DATA_DIR = "/kaggle/working/data"
train_df = pd.read_parquet(f"{DATA_DIR}/train.parquet")

# Dùng cấu hình tương đồng với mô hình Baseline thắng (max_features=20000, ngram_range=(1, 2), min_df=5)
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95,
    sublinear_tf=True,
    lowercase=True,
)

X_tfidf = vectorizer.fit_transform(train_df["text"].astype(str))
vocab = np.array(vectorizer.get_feature_names_out())
stars = train_df["stars"].to_numpy(dtype=int)

top_k = 12
distinctive_rows = []
high_tfidf_rows = []

for s in [1, 2, 3, 4, 5]:
    mask_s = stars == s
    mask_other = ~mask_s

    mean_s = np.asarray(X_tfidf[mask_s].mean(axis=0)).ravel()
    mean_other = np.asarray(X_tfidf[mask_other].mean(axis=0)).ravel()
    diff_score = mean_s - mean_other

    # Top từ/cụm từ đặc thù nhất cho mức sao s (so với các mức sao khác)
    top_diff_idx = np.argsort(diff_score)[::-1][:top_k]
    distinctive_rows.append({
        "stars": s,
        "n_train": int(mask_s.sum()),
        "top_distinctive_ngrams": ", ".join(
            f"{vocab[i]} ({diff_score[i]:+.4f})" for i in top_diff_idx
        ),
    })

    # Top từ/cụm từ có trọng số TF-IDF trung bình cao nhất trong mức sao s
    top_mean_idx = np.argsort(mean_s)[::-1][:top_k]
    high_tfidf_rows.append({
        "stars": s,
        "n_train": int(mask_s.sum()),
        "top_mean_tfidf_ngrams": ", ".join(
            f"{vocab[i]} ({mean_s[i]:.4f})" for i in top_mean_idx
        ),
    })

print("--- 1. TỪ / CỤM 2 TỪ ĐẶC THÙ NHẤT THEO TỪNG MỨC SAO (DIFF MEAN TF-IDF) ---")
display(pd.DataFrame(distinctive_rows).set_index("stars"))

print("\n--- 2. TỪ / CỤM 2 TỪ CÓ TRỌNG SỐ TF-IDF TRUNG BÌNH CAO NHẤT THEO MỨC SAO ---")
display(pd.DataFrame(high_tfidf_rows).set_index("stars"))

# Tách riêng xem 2-gram (cụm 2 từ) nổi bật nhất theo từng mức sao
bigram_mask = np.array([len(term.split()) == 2 for term in vocab])
bigram_vocab = vocab[bigram_mask]
X_bi = X_tfidf[:, bigram_mask]

bigram_rows = []
for s in [1, 2, 3, 4, 5]:
    mask_s = stars == s
    mask_other = ~mask_s
    bi_diff = np.asarray(X_bi[mask_s].mean(axis=0)).ravel() - np.asarray(X_bi[mask_other].mean(axis=0)).ravel()
    top_bi_idx = np.argsort(bi_diff)[::-1][:10]
    bigram_rows.append({
        "stars": s,
        "top_2grams": ", ".join(f"{bigram_vocab[i]} ({bi_diff[i]:+.4f})" for i in top_bi_idx),
    })

print("\n--- 3. TOP 10 CỤM 2 TỪ (2-GRAM) ĐẶC THÙ THEO MỨC SAO ---")
display(pd.DataFrame(bigram_rows).set_index("stars"))

--- 1. TỪ / CỤM 2 TỪ ĐẶC THÙ NHẤT THEO TỪNG MỨC SAO (DIFF MEAN TF-IDF) ---


,n_train,top_distinctive_ngrams
stars,,
1,6427,"not (+0.0133), no (+0.0130), told (+0.0122), n..."
2,3270,"not (+0.0132), was (+0.0093), but (+0.0092), o..."
3,4157,"but (+0.0149), good (+0.0101), not (+0.0088), ..."
4,8730,"good (+0.0110), but (+0.0059), little (+0.0057..."
5,19416,"great (+0.0123), amazing (+0.0105), best (+0.0..."



--- 2. TỪ / CỤM 2 TỪ CÓ TRỌNG SỐ TF-IDF TRUNG BÌNH CAO NHẤT THEO MỨC SAO ---


,n_train,top_mean_tfidf_ngrams
stars,,
1,6427,"the (0.0410), to (0.0383), and (0.0347), was (..."
2,3270,"the (0.0447), was (0.0382), and (0.0351), to (..."
3,4157,"the (0.0458), was (0.0361), and (0.0353), to (..."
4,8730,"the (0.0454), and (0.0394), was (0.0307), to (..."
5,19416,"the (0.0437), and (0.0428), to (0.0297), is (0..."



--- 3. TOP 10 CỤM 2 TỪ (2-GRAM) ĐẶC THÙ THEO MỨC SAO ---


,top_2grams
stars,
1,"do not (+0.0066), the worst (+0.0065), will ne..."
2,"was not (+0.0052), to be (+0.0045), but the (+..."
3,"was good (+0.0064), good but (+0.0061), but th..."
4,"very good (+0.0043), was good (+0.0032), good ..."
5,"the best (+0.0079), highly recommend (+0.0060)..."


In [9]:
from pathlib import Path
import pandas as pd
from src.baseline import fit_baseline, predict_baseline, save_baseline
from src.metrics import compute_metrics, per_star_metrics, threshold_moc1, mean_predictor_metrics

train_df = pd.read_parquet(f"{DATA_DIR}/train.parquet")
val_df   = pd.read_parquet(f"{DATA_DIR}/val.parquet")
test_df  = pd.read_parquet(f"{DATA_DIR}/test.parquet")

# 1. Mốc dự đoán trung bình (Mean Predictor)
mean_m = mean_predictor_metrics(train_df["stars"], float(train_df["stars"].mean()))
print(f"Mean Predictor Val RMSE: {mean_m['rmse']:.4f} | MAE: {mean_m['mae']:.4f}")
# 2. Chạy Grid Search 84 tổ hợp Baseline (TF-IDF + Ridge)
res = fit_baseline(train_df, val_df, seed=42)
val_pred = predict_baseline(res["pipeline"], val_df["text"].tolist())
val_m = compute_metrics(val_df["stars"], val_pred)
val_ps = per_star_metrics(val_df["stars"], val_pred)

print("Best params:", res["best_params"])
print(f"Baseline Val RMSE: {val_m['rmse']:.4f} | MAE: {val_m['mae']:.4f} | Macro-MAE: {val_ps['macro_mae']:.4f}")
print(f"Ngưỡng mục tiêu Mốc 1 (0.85 * RMSE_val): {threshold_moc1(val_m['rmse']):.4f}")

# 3. Lưu artifact và dự đoán Test của Baseline
PREDS_DIR = Path("/kaggle/working/results/preds")
PREDS_DIR.mkdir(parents=True, exist_ok=True)
save_baseline(res, PREDS_DIR / "baseline_val.pkl", overwrite=True)

test_pred = predict_baseline(res["pipeline"], test_df["text"].tolist())
pd.DataFrame({
    "review_id": test_df["review_id"],
    "true_stars": test_df["stars"].to_numpy(dtype=int),
    "pred_baseline": test_pred
}).to_csv(PREDS_DIR / "baseline_test.csv", index=False)
print("Đã lưu xong baseline_val.pkl và baseline_test.csv!")

Mean Predictor Val RMSE: 1.4787 | MAE: 1.2616
Best params: {'max_features': np.int64(20000), 'ngram_range': (1, 2), 'min_df': np.int64(5), 'alpha': np.float64(1.0)}
Baseline Val RMSE: 0.7792 | MAE: 0.5758 | Macro-MAE: 0.6329
Ngưỡng mục tiêu Mốc 1 (0.85 * RMSE_val): 0.6623
Đã lưu xong baseline_val.pkl và baseline_test.csv!
